In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from transformers import AutoTokenizer, BertModel, BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split


# STEP 2: Load & Split Dataset (70/15/15)%

# Reads CSV with separator (;), assuming the labeled dataset is 'comments_labeled.csv'
# Comments are processed in comment processor.ipynb
df = pd.read_csv('comments_labeled.csv', sep=';')

# Builds combined aspect and sentiment column untuk for stratification [cite: 105, 130]
df['stratify_col'] = df['aspect_class'].astype(str) + "_" + df['sentiment_class'].astype(str)

# Filter extreme combination class with only 1 member to preserve stratified split
class_counts = df['stratify_col'].value_counts()
rare_classes = class_counts[class_counts < 2].index.tolist()

if len(rare_classes) > 0:
    print(f"[Info] Deleting {len(rare_classes)} rare combination classes: {rare_classes}")
    df = df[~df['stratify_col'].isin(rare_classes)]

# Step 1 data distribution (70% Train, 30% combined Val & Test) [cite: 123, 130, 165]
train_df, val_test_df = train_test_split(
    df, 
    test_size=0.30, 
    random_state=41, # Fixed random seed for reproducibility 
    stratify=df['stratify_col'] # [cite: 130, 165]
)

# Step 2 data distribution (15% Validation, 15% Testing from total dataset) [cite: 123, 130, 165]
val_test_counts = val_test_df['stratify_col'].value_counts()
rare_val_test = val_test_counts[val_test_counts < 2].index.tolist()

if len(rare_val_test) > 0:
    val_df, test_df = train_test_split(val_test_df, test_size=0.50, random_state=41)
else:
    val_df, test_df = train_test_split(val_test_df, test_size=0.50, random_state=41, stratify=val_test_df['stratify_col'])

# Delete stratification column helper
train_df = train_df.drop(columns=['stratify_col'])
val_df = val_df.drop(columns=['stratify_col'])
test_df = test_df.drop(columns=['stratify_col'])

print(f"[Success!] Dataset Split -> Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}\n")

In [ ]:
# STEP 3: Prepare Dataset & Tokenizer

MAX_LEN = 128 # [cite: 131]
BATCH_SIZE = 16 # [cite: 131]
LR = 2e-5 # [cite: 131]
EPOCHS = 4 # [cite: 131]

# Choose model to test [cite: 86]
# MODEL_NAME = "indobenchmark/indobert-base-p1"       # IndoBERT P1
MODEL_NAME = "indobenchmark/indobert-base-p2"     # IndoBERT P2 (optimized and more social media data)

print(f"Loading tokenizer for {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class YouTubeDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

        # Sets aspect map and sentiment map
        self.aspect_map = {
            'game_general': 0,
            'gameplay': 1, 
            'graphics': 2, 
            'story': 3, 
            'streamer': 4, 
            'other': 5,
            'off_topic': 5 
        }
        
        self.sentiment_map = {
            'positive': 0, 
            'negative': 1, 
            'neutral': 2
        }
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, index):
        row = self.data.iloc[index]
        comment = str(row['comment_text'])
        
        inputs = self.tokenizer(
            comment,
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_token_type_ids=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'token_type_ids': inputs['token_type_ids'].flatten(),
            'aspect_label': torch.tensor(self.aspect_map[row['aspect_class']], dtype=torch.long),
            'sentiment_label': torch.tensor(self.sentiment_map[row['sentiment_class']], dtype=torch.long)
        }

train_dataset = YouTubeDataset(train_df, tokenizer, MAX_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
# STEP 4: ARCHITECTURE & FINE-TUNING LOOP

class ABSAMultiTaskModel(nn.Module):
    def __init__(self, model_name, num_aspects=6, num_sentiments=3):
        super(ABSAMultiTaskModel, self).__init__()
        
        self.bert = BertModel.from_pretrained(model_name, ignore_mismatched_sizes=True)
        hidden_size = self.bert.config.hidden_size
        
        self.aspect_classifier = nn.Linear(hidden_size, num_aspects)
        self.sentiment_classifier = nn.Linear(hidden_size, num_sentiments)
        
    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled_output = outputs[1]
        
        aspect_logits = self.aspect_classifier(pooled_output)
        sentiment_logits = self.sentiment_classifier(pooled_output)
        
        return aspect_logits, sentiment_logits

# Setup Environment Pytorch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ABSAMultiTaskModel(MODEL_NAME).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR) # Using Adam Optimizer [cite: 131]
criterion = nn.CrossEntropyLoss()

print(f"\n[Start] Initiating ({MODEL_NAME}) training on: {device}")

history = {
    'train_loss': [],
    'aspect_accuracy': [],
    'sentiment_accuracy': []
}

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    # Local variable to calculate accuract on an epoch
    correct_aspect = 0
    correct_sentiment = 0
    total_samples = 0
    
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        token_type = batch['token_type_ids'].to(device)
        
        aspect_labels = batch['aspect_label'].to(device)
        sentiment_labels = batch['sentiment_label'].to(device)
        
        aspect_logits, sentiment_logits = model(input_ids, mask, token_type)
        
        # Count loss multi-task
        loss_aspect = criterion(aspect_logits, aspect_labels)
        loss_sentiment = criterion(sentiment_logits, sentiment_labels)
        loss = loss_aspect + loss_sentiment
        
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        # Calculate training accuracy real-time
        aspect_preds = torch.argmax(aspect_logits, dim=1)
        sentiment_preds = torch.argmax(sentiment_logits, dim=1)
        
        correct_aspect += (aspect_preds == aspect_labels).sum().item()
        correct_sentiment += (sentiment_preds == sentiment_labels).sum().item()
        total_samples += aspect_labels.size(0)
        
        if step % 20 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} | Step {step}/{len(train_loader)} | Contemporary Loss: {loss.item():.4f}")
            
    print(f"=== Epoch Done {epoch+1} -> Average Loss for this Epoch: {total_loss/len(train_loader):.4f} ===\n")

    # Calculate average metric after epoch
    epoch_loss = total_loss / len(train_loader)
    epoch_asp_acc = correct_aspect / total_samples
    epoch_sent_acc = correct_sentiment / total_samples
    
    # Save to history
    history['train_loss'].append(epoch_loss)
    history['aspect_accuracy'].append(epoch_asp_acc)
    history['sentiment_accuracy'].append(epoch_sent_acc)
    
    print(f"\n=== Epoch Done {epoch+1}/{EPOCHS} ===")
    print(f"Average Loss    : {epoch_loss:.4f}")
    print(f"Aspect Accuracy   : {epoch_asp_acc*100:.2f}%")
    print(f"Aspect Sentiment  : {epoch_sent_acc*100:.2f}%\n")

In [ ]:
import matplotlib.pyplot as plt

# Set theme plot
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']

epochs_range = range(1, EPOCHS + 1)

fig, ax = plt.subplots(figsize=(7, 5))

# Build plot for aspect and sentiment accuracy
ax.plot(epochs_range, history['aspect_accuracy'], marker='o', linewidth=2, color='#4ca3dd', label='Aspect Accuracy')
ax.plot(epochs_range, history['sentiment_accuracy'], marker='s', linewidth=2, linestyle='--', color='#f2994a', label='Sentiment Accuracy')

# Set label and title
ax.set_xlabel('Epoch', fontsize=11, fontweight='bold')
ax.set_ylabel('Accuracy Value', fontsize=11, fontweight='bold')
ax.set_title(f'Training Accuracy Improvement Graphic per Epoch\n({MODEL_NAME.split("/")[-1]})', fontsize=12, fontweight='bold', pad=15)
ax.set_xticks(epochs_range)
ax.set_ylim(0, 1.0) # Accuracy scale from 0 to 1 (100%)
ax.legend(loc='lower right', frameon=True)

# Add text value annotation on every coordinate point marker
for i, (asp_acc, sent_acc) in enumerate(zip(history['aspect_accuracy'], history['sentiment_accuracy'])):
    ax.annotate(f'{asp_acc*100:.1f}%', (epochs_range[i], asp_acc), textcoords="offset points", xytext=(0,10), ha='center', fontsize=9, fontweight='bold', color='#2d7bb0')
    ax.annotate(f'{sent_acc*100:.1f}%', (epochs_range[i], sent_acc), textcoords="offset points", xytext=(0,-15), ha='center', fontsize=9, fontweight='bold', color='#d07421')

plt.tight_layout()
plt.savefig('training_accuracy_history.png', dpi=300)
plt.show()

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

# STEP 5: Prepare Dataloader Test

# Use test_df from previous stratified split 
test_dataset = YouTubeDataset(test_df, tokenizer, MAX_LEN)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

def evaluate_model(model, dataloader, device, num_aspect_classes=6, num_sentiment_classes=3):
    model.eval() # Switch to evaluation mode
    
    all_aspect_preds = []
    all_aspect_labels = []
    all_aspect_probs = []
    
    all_sentiment_preds = []
    all_sentiment_labels = []
    all_sentiment_probs = []
    
    with torch.no_grad(): # Deactivate gradient calculation
        for batch in dataloader:
            input_ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            token_type = batch['token_type_ids'].to(device)
            
            aspect_labels = batch['aspect_label'].to(device)
            sentiment_labels = batch['sentiment_label'].to(device)
            
            # Forward pass gets logits prediction
            aspect_logits, sentiment_logits = model(input_ids, mask, token_type)
            
            # Calculates probability (for AUC-ROC) using Softmax
            aspect_prob = F.softmax(aspect_logits, dim=1)
            sentiment_prob = F.softmax(sentiment_logits, dim=1)
            
            # Gets the class with the highest probability score
            aspect_preds = torch.argmax(aspect_logits, dim=1)
            sentiment_preds = torch.argmax(sentiment_logits, dim=1)
            
            # Save results to list (relocate to CPU)
            all_aspect_preds.extend(aspect_preds.cpu().numpy())
            all_aspect_labels.extend(aspect_labels.cpu().numpy())
            all_aspect_probs.extend(aspect_prob.cpu().numpy())
            
            all_sentiment_preds.extend(sentiment_preds.cpu().numpy())
            all_sentiment_labels.extend(sentiment_labels.cpu().numpy())
            all_sentiment_probs.extend(sentiment_prob.cpu().numpy())
            
    # Convert to numpy array for scikit-learn calculation
    all_aspect_labels = np.array(all_aspect_labels)
    all_aspect_preds = np.array(all_aspect_preds)
    all_aspect_probs = np.array(all_aspect_probs)
    
    all_sentiment_labels = np.array(all_sentiment_labels)
    all_sentiment_preds = np.array(all_sentiment_preds)
    all_sentiment_probs = np.array(all_sentiment_probs)
    
    # Calculate Metric Evaluation (macro Average)
    # Apply macro average in to balance rare classes
    acc_asp = accuracy_score(all_aspect_labels, all_aspect_preds)
    p_asp, r_asp, f1_asp, _ = precision_recall_fscore_support(all_aspect_labels, all_aspect_preds, average='macro', zero_division=0)
    
    acc_sent = accuracy_score(all_sentiment_labels, all_sentiment_preds)
    p_sent, r_sent, f1_sent, _ = precision_recall_fscore_support(all_sentiment_labels, all_sentiment_preds, average='macro', zero_division=0)
    
    # Calculate AUC-ROC Multiclass (One-vs-Rest)
    try:
        auc_asp = roc_auc_score(all_aspect_labels, all_aspect_probs, multi_class='ovr', average='macro')
    except ValueError:
        auc_asp = np.nan
        
    try:
        auc_sent = roc_auc_score(all_sentiment_labels, all_sentiment_probs, multi_class='ovr', average='macro')
    except ValueError:
        auc_sent = np.nan

    # Show metric evaluation results
    print("="*20 + f" MODEL Evaluation Results For ({MODEL_NAME})" + "="*20)
    print(f"\n--- ASPECT CLASSIFICATION METRICS ---")
    print(f"Accuracy  : {acc_asp:.4f}")
    print(f"Precision : {p_asp:.4f}")
    print(f"Recall    : {r_asp:.4f}")
    print(f"F1-Score  : {f1_asp:.4f}")
    print(f"AUC-ROC   : {auc_asp if np.isnan(auc_asp) else f'{auc_asp:.4f}'}")
    
    print(f"\n--- SENTIMENT CLASSIFICATION METRICS ---")
    print(f"Accuracy  : {acc_sent:.4f}")
    print(f"Precision : {p_sent:.4f}")
    print(f"Recall    : {r_sent:.4f}")
    print(f"F1-Score  : {f1_sent:.4f}")
    print(f"AUC-ROC   : {auc_sent if np.isnan(auc_sent) else f'{auc_sent:.4f}'}")
    print("="*62)
    
    return {
        'aspect_actual': all_aspect_labels, 'aspect_pred': all_aspect_preds,
        'sentiment_actual': all_sentiment_labels, 'sentiment_pred': all_sentiment_preds
    }

# Run evaluation on active hardware (GPU/CPU)
results = evaluate_model(model, test_loader, device)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# STEP 6: Visualize results

# Map all indexes to its actual class names for diagram readibility
aspect_labels_map = {0: 'Game_General', 1: 'Gameplay', 2: 'Graphics', 3: 'Story', 4: 'Streamer', 5: 'Other'}
sentiment_labels_map = {0: 'Positive', 1: 'Negative', 2: 'Neutral'}

# Build DataFrame from testing (using 'results' veriable from evaluation phase)
df_viz = pd.DataFrame({
    'Aspect': [aspect_labels_map[a] for a in results['aspect_pred']],
    'Sentiment': [sentiment_labels_map[s] for s in results['sentiment_pred']]
})

# Set plot theme color
colors_aspect = sns.color_palette('pastel')[0:6]
colors_sentiment = ['#99ff99', '#ff9999', '#ffcc99'] 

# 1st visualization: Aspect Distribution (full diagram)

plt.figure(figsize=(8, 8))
aspect_counts = df_viz['Aspect'].value_counts()

plt.pie(
    aspect_counts, 
    labels=aspect_counts.index, 
    autopct='%1.1f%%', 
    startangle=140, 
    colors=colors_aspect,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1}
)
plt.title("Aspect Distribution Ratio on Indonesian YouTube Gaming Comments", fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('aspect_distribution.png', dpi=300) # Save to local
plt.show()

# 2nd visualization: Sentiment per Aspect Class (sub-diagram)

unique_aspects = df_viz['Aspect'].unique()
print("\n--- Processing Sentiment per Aspect Class Graph ---")

for aspect in unique_aspects:
    # Filter data berdasarkan aspek spesifik
    df_sub = df_viz[df_viz['Aspect'] == aspect]
    sentiment_counts = df_sub['Sentiment'].value_counts()
    
    plt.figure(figsize=(5, 5))
    plt.pie(
        sentiment_counts, 
        labels=sentiment_counts.index, 
        autopct='%1.1f%%', 
        startangle=90, 
        colors=[colors_sentiment[sentiment_labels_map.get(s, 2)] for s in sentiment_counts.index],
        wedgeprops={'edgecolor': 'white', 'linewidth': 1}
    )
    plt.title(f"Sentiment Feedback for Aspect: {aspect}", fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'sentiment_ratio_{aspect.lower().replace("/", "_")}.png', dpi=300)
    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_curve, auc

# Set new theme again
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman'] + plt.rcParams['font.serif']

# 3rd visualization: F1-score Bar Graphic

# Extract data from variable results
# Ensure that all prediction and actual data is globally unpacked
y_true_asp = results['aspect_actual']
y_pred_asp = results['aspect_pred']
y_true_sent = results['sentiment_actual']
y_pred_sent = results['sentiment_pred']

# Recount Macro F1-Score globally
_, _, f1_asp_global, _ = precision_recall_fscore_support(y_true_asp, y_pred_asp, average='macro', zero_division=0)
_, _, f1_sent_global, _ = precision_recall_fscore_support(y_true_sent, y_pred_sent, average='macro', zero_division=0)

fig, ax = plt.subplots(figsize=(6, 4.5))
categories = ['Aspect Classification', 'Sentiment Classification']
f1_scores = [f1_asp_global, f1_sent_global] # Takes the F1 variable from evaluation

bars = ax.bar(categories, f1_scores, color=['#4ca3dd', '#f2994a'], width=0.4)

ax.set_ylabel('Macro F1-Score', fontsize=12, fontweight='bold')
ax.set_title(f'Macro F1-Score Performance Graph ({MODEL_NAME.split("/")[-1]})', fontsize=12, fontweight='bold', pad=15)
ax.set_ylim(0, 1.0)

# Add number label on every bar in graph
for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:.4f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3), textcoords="offset points",
                ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('f1_score_performance.png', dpi=300)
plt.show()


# 4th visualization: Multi-Class ROC Curves ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# ROC Plotting for Aspect Classification (6 classes)
# all_aspect_labels & all_aspect_probs taken from evaluation result array
for i in range(6):
    fpr, tpr, _ = roc_curve(y_true_asp == i, y_pred_asp == i)
    ax1.plot(fpr, tpr, lw=2, label=f'Class {i} (AUC = {auc(fpr, tpr):.2f})')
ax1.plot([0, 1], [0, 1], 'k--', lw=1.5)
ax1.set_title(f'ROC Curve: Aspect Classification', fontweight='bold', fontsize=11)
ax1.set_xlabel('False Positive Rate', fontsize=10)
ax1.set_ylabel('True Positive Rate', fontsize=10)
ax1.legend(loc='lower right', fontsize=9)

# ROC Plotting for Sentiment Classification (3 classes)
# all_sentiment_labels & all_sentiment_probs taken from evaluation result array
for i in range(3):
    fpr, tpr, _ = roc_curve(y_true_sent == i, y_pred_sent == i)
    ax2.plot(fpr, tpr, lw=2, label=f'Class {i} (AUC = {auc(fpr, tpr):.2f})')
ax2.plot([0, 1], [0, 1], 'k--', lw=1.5)
ax2.set_title(f'ROC Curve: Sentiment Classification', fontweight='bold', fontsize=11)
ax2.set_xlabel('False Positive Rate', fontsize=10)
ax2.set_ylabel('True Positive Rate', fontsize=10)
ax2.legend(loc='lower right', fontsize=9)

plt.suptitle('Receiver Operating Characteristic (ROC) Curves Multi-Task Learning', fontsize=13, fontweight='bold', y=0.98)
plt.tight_layout()
plt.savefig('roc_curves_evaluation.png', dpi=300)
plt.show()
print ("Class 0 : Game_General, Class 1 : Gameplay, Class 2 : Graphics")
print ("Class 3 : Story, Class 4 : Streamer, Class 5 : Other/Off-Topic")